# Homework 14 - Trent Douglas

## Initial Conditions

Reference trajectory will be provided (includes J2000, ECEF, and Sun-Vehicle-Earth angle).

**Epoch:** 06/01/2012 00:00:00  

**State Vector & Orbital Elements:**

- Semi-major axis (A): 26560020.995 m  
- Eccentricity (E): 0.001673  
- Inclination (I): 55.580348 deg  
- RAAN: 53.638749 deg  
- Argument of Perigee (ω): 40.335486 deg  
- True Anomaly (ν): 265.154528 deg  
- Mean Anomaly (MA): 265.345602 deg  
- Argument of Latitude: 305.490014 deg  

**Position (J2000):**
- X: 18988410.689 m  
- Y: 5170910.062 m  
- Z: -17841865.424 m  

**Velocity (J2000):**
- XD: 841.572707 m/s  
- YD: 3292.087758 m/s  
- ZD: 1859.379002 m/s  


Starting at **06/01/2012 00:00:00**, and using a **Time of Flight approach**, estimate the times of the next **five (5)**:

1. Node crossings (use J2000 vectors)  
2. Perigee passages  
3. Eclipse centers (orbit midnight)  

For each eclipse:

- Estimate the **duration of the eclipse**
- You may need to compute a **new sun vector** for each estimate
- Verify your results against the **tabulated data**

In [ ]:
import csv
from typing import List
from standards import *

J2K_POS = Vector3(18988410.689, 5170910.062, -17841865.424)
J2K_VEL = Vector3(841.572707, 3292.087758, 1859.379002)
KEP = KeplerianElements(J2K_POS, J2K_VEL)
SMA = KEP.a
ECC = KEP.ecc
INC = KEP.inc_deg
RAAN = KEP.raan_deg
ARGP = KEP.argp_deg
TA = KEP.ta_deg
MA = 265.345602
ARG_LAT = 305.490014
Start_Time = datetime(2012,6,1,0,0,0,0)

### nodal crossings:
print("Estimated First 5 Nodal Crossings:")
for i in range(5):
    if i == 0:
        TA_Node_rad = 2*pi-radians(ARGP)
        TA_SV_rad = radians(TA)
        E_SV_rad = compute_eccentric_anomaly(TA_SV_rad, ECC)
        E_Node_rad = compute_eccentric_anomaly(TA_Node_rad, ECC)
        Orbital_Period = compute_period(SMA, KEP.mu_earth)
        Mean_Motion = compute_mean_motion(KEP.mu_earth, SMA)
        k = 0
        if TA_SV_rad > TA_Node_rad: k = 1
        time_from_sv_to_node = k*Orbital_Period + 1/Mean_Motion*(E_Node_rad-ECC*sin(E_Node_rad)) - 1/Mean_Motion*(E_SV_rad-ECC*sin(E_SV_rad))
        final_time = Start_Time + timedelta(seconds = time_from_sv_to_node)
    else:
        final_time = prev_time + timedelta(seconds=Orbital_Period)
    prev_time = final_time
    print(final_time.strftime('%d-%b-%Y %H:%M:%S'))

### perigee crossings:   
print("\nEstimated First 5 Perigee Crossings:")
for i in range(5):
    if i == 0:
        TA_Perigee_rad = 0
        TA_SV_rad = radians(TA)
        E_SV_rad = compute_eccentric_anomaly(TA_SV_rad, ECC)
        E_Perigee_rad = compute_eccentric_anomaly(TA_Perigee_rad, ECC)
        Orbital_Period = compute_period(SMA, KEP.mu_earth)
        Mean_Motion = compute_mean_motion(KEP.mu_earth, SMA)
        time_from_sv_to_perigee = 1/Mean_Motion*(2*pi-(E_SV_rad-ECC*sin(E_SV_rad)))
        final_time = Start_Time + timedelta(seconds = time_from_sv_to_perigee)
    else:
        final_time = prev_time + timedelta(seconds=Orbital_Period)
    prev_time = final_time
    print(final_time.strftime('%d-%b-%Y %H:%M:%S'))

### eclipses:
print()
print("Estimated First 5 midnights:")
print(f"{'#':<4} {'Start':>24} {'Midpoint':>24} {'End':>24} {'Beta':>10} {'Duration':>10}")
temp_start_time = Start_Time

for i in range(5):
    if i == 0:
        jDate = compute_j_date(temp_start_time.year, temp_start_time.month, temp_start_time.day, temp_start_time.hour, temp_start_time.minute, temp_start_time.second)
        SUN_Vector = compute_sun_vector(jDate)
        SUN_Vector = SUN_Vector/SUN_Vector.magnitude()
        h = compute_angular_momentum_vector(J2K_POS, J2K_VEL)
        h = h/h.magnitude()
        R_perifocal_ECI = np.array([
            [math.cos(KEP.raan)*math.cos(KEP.argp)-math.sin(KEP.raan)*math.sin(KEP.argp)*math.cos(KEP.inc), -math.cos(KEP.raan)*math.sin(KEP.argp)-math.sin(KEP.raan)*math.cos(KEP.argp)*math.cos(KEP.inc), math.sin(KEP.raan)*math.sin(KEP.inc)],
            [math.sin(KEP.raan)*math.cos(KEP.argp)+math.cos(KEP.raan)*math.sin(KEP.argp)*math.cos(KEP.inc), -math.sin(KEP.raan)*math.sin(KEP.argp)+math.cos(KEP.raan)*math.cos(KEP.argp)*math.cos(KEP.inc), -math.cos(KEP.raan)*math.sin(KEP.inc)],
            [math.sin(KEP.inc)*math.sin(KEP.argp), math.sin(KEP.inc)*math.cos(KEP.argp), math.cos(KEP.inc)]
        ])
        Sun_perifocal = np.linalg.inv(R_perifocal_ECI) @ SUN_Vector.get_np_vector()
        noon_vector = Vector3(Sun_perifocal[0][0], Sun_perifocal[1][0], 0)
        midnight_vector = -1*noon_vector
        ta_noon_rad = atan2(noon_vector.y, noon_vector.x)
        ta_midnight_rad = pi+ta_noon_rad
        if ta_midnight_rad < 0: ta_midnight_rad = ta_midnight_rad+2*pi
        k = 0
        if KEP.ta > ta_midnight_rad: k = 1
        Orbital_Period = compute_period(SMA, KEP.mu_earth)
        E_SV_rad = compute_eccentric_anomaly(KEP.ta, ECC)
        E_Midnight_rad = compute_eccentric_anomaly(ta_midnight_rad, ECC)
        Orbital_Period = compute_period(SMA, KEP.mu_earth)
        Mean_Motion = compute_mean_motion(KEP.mu_earth, SMA)
        time_from_sv_to_midnight = k*Orbital_Period + 1/Mean_Motion*(E_Midnight_rad-ECC*sin(E_Midnight_rad)) - 1/Mean_Motion*(E_SV_rad-ECC*sin(E_SV_rad))
        midnight_time = temp_start_time+timedelta(seconds = time_from_sv_to_midnight)
        Earth_radius = 6378137 #m
        p = asin(Earth_radius / J2K_POS.magnitude())
        e = 0
        midnight_jDate = compute_j_date(midnight_time.year, midnight_time.month, midnight_time.day, midnight_time.hour, midnight_time.minute, midnight_time.second)
        SUN_Vector_midnight = compute_sun_vector(midnight_jDate)
        S_hat_midnight = SUN_Vector_midnight / SUN_Vector_midnight.magnitude()
        beta = asin(S_hat_midnight.dot(h))
        if beta < p: e = e*acos(cos(p)/cos(beta))
        umbral_eclipse_duration = Orbital_Period/pi * acos(cos(p)/cos(beta))
        entry = (midnight_time - timedelta(seconds=umbral_eclipse_duration)/2)
        exit = (midnight_time + timedelta(seconds=umbral_eclipse_duration)/2)
    else:
        p = asin(Earth_radius / J2K_POS.magnitude())
        midnight_time = midnight_time + timedelta(seconds=Orbital_Period)
        midnight_jDate = compute_j_date(midnight_time.year, midnight_time.month, midnight_time.day, midnight_time.hour, midnight_time.minute, midnight_time.second)
        SUN_Vector_midnight = compute_sun_vector(midnight_jDate)
        S_hat_midnight = SUN_Vector_midnight / SUN_Vector_midnight.magnitude()
        beta = asin(S_hat_midnight.dot(h))
        if beta < p: e = e*acos(cos(p)/cos(beta))
        umbral_eclipse_duration = Orbital_Period/pi * acos(cos(p)/cos(beta))
        entry = midnight_time - timedelta(seconds=umbral_eclipse_duration/2)
        exit = midnight_time + timedelta(seconds=umbral_eclipse_duration/2)

    print(f"{i+1:<4} {entry.strftime('%d-%b-%Y %H:%M:%S'):>24} {midnight_time.strftime('%d-%b-%Y %H:%M:%S'):>24} {exit.strftime('%d-%b-%Y %H:%M:%S'):>24} {degrees(beta):>10.3f} {umbral_eclipse_duration/60:>10.3f}")

Estimated First 5 Nodal Crossings:
01-Jun-2012 01:48:34
01-Jun-2012 13:46:32
02-Jun-2012 01:44:30
02-Jun-2012 13:42:28
03-Jun-2012 01:40:25

Estimated First 5 Perigee Crossings:
01-Jun-2012 03:08:46
01-Jun-2012 15:06:44
02-Jun-2012 03:04:42
02-Jun-2012 15:02:39
03-Jun-2012 03:00:37

Estimated First 5 midnights:
#                       Start                 Midpoint                      End       Beta   Duration
1        01-Jun-2012 08:13:04     01-Jun-2012 08:40:47     01-Jun-2012 09:08:29      0.082     55.413
2        01-Jun-2012 20:11:02     01-Jun-2012 20:38:44     01-Jun-2012 21:06:27     -0.253     55.405
3        02-Jun-2012 08:09:01     02-Jun-2012 08:36:42     02-Jun-2012 09:04:23     -0.588     55.366
4        02-Jun-2012 20:07:01     02-Jun-2012 20:34:40     02-Jun-2012 21:02:19     -0.922     55.294
5        03-Jun-2012 08:05:02     03-Jun-2012 08:32:38     03-Jun-2012 09:00:14     -1.257     55.191
